# finance / portfolio / allocation

Share of the portfolio to hold in risky assets at a given volatility, capped at fully invested.

| Term | In This Notebook |
|---|---|
| **Intent** | Give the Merton share, the excess return over the risk aversion times the variance, so the risky share follows the volatility you expect. Placeholder: the single coefficient is recovered from synthetic points |
| **Project Flavour** | `pyfunc`, served by `mlserver` (`model.md` 3.2) |
| **Data Splits** | `training` ten volatilities with the risky share; `validation` five further; `backtest` ten across the range including capped ones, held back for acceptance; `golden` the exact volatilities that become `golden.json` |

## Setup

- **Stage:** none, preparation for the stages below.
- **Intent:** put `workbench` on the path, load `env`, and declare the model name, dataset names and signature names as constants (`model.md` 6.6).

In [ ]:
import json
from os.path import join

from workbench.config import NOTEBOOK_DIR
from workbench.data import ensure_data, golden_data
from workbench.env import init_env
from workbench.project import provenance, record_project, scratch_project_dir

env = init_env()

WORKSPACE = "finance/portfolio/allocation"
MODEL_NAME = "finance_portfolio_allocation"
TARGETS = [{"kind": "served"}]
TRAINING = "portfolio_allocation_training_2026-11-05_2026-11-05"
VALIDATION = "portfolio_allocation_validation_2026-11-05_2026-11-05"
BACKTEST = "portfolio_allocation_backtest_2026-11-05_2026-11-05"
GOLDEN = "portfolio_allocation_golden"
INPUTS = ["finance_portfolio_allocation_volatility_pct"]
OUTPUTS = ["finance_portfolio_allocation_risky_pct"]

## Capture Data

- **Stage:** 1, capture data (`model.md` 7.1).
- **Intent:** write the `training`, `validation` and `backtest` files only when they are absent (`model.md` 6.5). The source is synthetic, with a coefficient of 1.5, so no query runs.

In [ ]:
def share(volatility, coefficient=1.5):
    return min(100.0, coefficient / (volatility / 100.0) ** 2)


def labelled(volatilities):
    return {INPUTS[0]: volatilities, OUTPUTS[0]: [round(share(volatility), 6) for volatility in volatilities]}


training_pl, training_path = ensure_data(WORKSPACE, "training", TRAINING, lambda: labelled([13.0, 14.0, 15.0, 16.0, 18.0, 20.0, 22.0, 25.0, 28.0, 30.0]))
validation_pl, validation_path = ensure_data(WORKSPACE, "validation", VALIDATION, lambda: labelled([13.5, 17.0, 21.0, 26.0, 29.0]))
backtest_pl, backtest_path = ensure_data(WORKSPACE, "backtest", BACKTEST, lambda: labelled([5.0, 8.0, 10.0, 11.0, 12.0, 35.0, 40.0, 45.0, 50.0, 60.0]))
print("training", training_pl.height, "validation", validation_pl.height, "backtest", backtest_pl.height)

## Fit

- **Stage:** 3, fit.
- **Intent:** fit the coefficient, excess return over risk aversion, by least squares through the origin on `training` only, where no share is capped (R6). The two are not separable from this data, which the plan records.

In [ ]:
import numpy as np

inverse_variance = 1.0 / (training_pl[INPUTS[0]].to_numpy() / 100.0) ** 2
COEFFICIENT = round(float(np.dot(inverse_variance, training_pl[OUTPUTS[0]].to_numpy()) / np.dot(inverse_variance, inverse_variance)), 4)
PARAMS = {"coefficient": COEFFICIENT}
print("fitted", PARAMS)
assert PARAMS == {"coefficient": 1.5}, "the fit did not recover the coefficient"

## Evaluate

- **Stage:** 4, evaluate.
- **Intent:** write `model.py` from the fitted values, build the signature, run it through the library runtime and check the `validation` and `backtest` error against the limit the notebook owns (R4, R14).

In [ ]:
from workbench.model import build_signature, evaluate_acceptance, measure, signature_input, signature_output
from workbench.replay import model_predict

MODEL_TEMPLATE = """import pandas as pd
from mlflow.models import set_model
from mlflow.pyfunc import PythonModel

INPUT = "finance_portfolio_allocation_volatility_pct"
OUTPUT = "finance_portfolio_allocation_risky_pct"
COEFFICIENT = {coefficient!r}


class Allocation(PythonModel):
    def predict(self, context, model_input: pd.DataFrame, params=None):
        return pd.DataFrame({{OUTPUT: [min(100.0, COEFFICIENT / (volatility / 100.0) ** 2) for volatility in model_input[INPUT]]}})


set_model(Allocation())
"""

model_path = join(NOTEBOOK_DIR, WORKSPACE, "model.py")
with open(model_path, "w") as model_file:
    model_file.write(MODEL_TEMPLATE.format(**PARAMS))

signature = build_signature(
    MODEL_NAME,
    [
        signature_input("finance_portfolio_allocation_volatility_pct", "percent", "Annualised volatility of the risky assets, percent."),
    ],
    [
        signature_output("finance_portfolio_allocation_risky_pct", "percent", "Share of the portfolio held in risky assets, 0 to 100 percent.", datatype="FP64"),
    ],
)
predict = model_predict(model_path, "pyfunc", signature)

In [ ]:
LIMIT = 0.01
SPLITS = (("validation", validation_pl), ("backtest", backtest_pl))


def acceptance_checks():
    answers = {split: predict({INPUTS[0]: frame[INPUTS[0]].to_list()}) for split, frame in SPLITS}
    return [
        {"split": split, "output": output, "name": "rmse", "value": measure(frame[output].to_list(), answers[split][output])["rmse"], "limit": LIMIT, "direction": "max"}
        for split, frame in SPLITS
        for output in OUTPUTS
    ]


evaluation = evaluate_acceptance(acceptance_checks, {"validation": [validation_path], "backtest": [backtest_path]})
for entry in evaluation["checks"]:
    print(entry["split"], entry["output"], entry["name"], "measured", entry["value"], "passed", entry["passed"])
print("acceptance", "passed" if evaluation["passed"] else "failed, the shipped model is outside the limit")

## Package

- **Stage:** 5, package.
- **Intent:** write the golden file from the shipped runtime, copy `model.py` and the evaluation into a project, verify it and replay the golden through the library runtime (R1, R3, R14).

In [ ]:
def golden_samples():
    xs = [5.0, 10.0, 12.2, 15.0, 20.0, 30.0, 60.0]
    answers = predict({INPUTS[0]: xs})
    return {INPUTS[0]: [*xs, None], **{output: [*answers[output], None] for output in OUTPUTS}, "expect": ["ok"] * len(xs) + ["error"]}

In [ ]:
from workbench.model import package_project, register_preview
from workbench.replay import require_replay

golden_pl, golden_path = golden_data(WORKSPACE, GOLDEN, golden_samples)

project_dir = scratch_project_dir(WORKSPACE)
manifest = package_project(
    project_dir,
    MODEL_NAME,
    "pyfunc",
    signature,
    golden_pl.iter_rows(named=True),
    provenance(
        "manual",
        "synthetic Merton shares, training split",
        [training_path, golden_path],
        "least squares coefficient through the origin",
        implementation="mlflow.pyfunc model.py, requires mlflow and pandas",
    ),
    model_path=model_path,
    evaluation=evaluation,
    sequence="from capped to well below fully invested",
)
print(json.dumps(manifest, indent=2))

require_replay(project_dir)
metadata = record_project(WORKSPACE, project_dir, TARGETS)
print("replayed the golden sequences exactly through the shipped runtime")

## Register

- **Stage:** 6, register.
- **Intent:** register the verified project with the alias `preview`. A person sets `release` after review and the parity gate (R5).

In [ ]:
version = register_preview(project_dir, MODEL_NAME, env["MLFLOW_TRACKING_URI"], TARGETS, metadata["snapshot_version"])
print("registered version", version)